# 1. Generate zombie clinical reports designed for feature vectors

This notebook **works backward**: starting from the structured table `zombie-health.csv`, it writes
synthetic clinical reports whose language is deliberately *controlled*, so that simple algorithms
(dictionary lookup + n-grams) can later turn every word or expression into a small **feature vector**.

Design rules for the generated text:

1. every report is organized in **sections** (Presentation, Negative Findings, Laboratory Analysis,
   Clinical History, Assessment);
2. every sentence states that **one** symptom or disease is **present** or **absent**;
3. diversity comes from rotating **templates** and **synonyms**, not from free writing;
4. in the default mode, a negation word always comes **before** the term it negates
   (`POST_NEGATION = True` turns on a "hard mode" that breaks this rule on purpose).

Besides the reports, the notebook saves the **ground truth** (which term is present or absent in
which sentence) and the **lexicon** (the "field guide" of known expressions) used by Notebook 2.

In [1]:
import random
import re
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CSV_PATH = Path("../../data/zombie/zombie-health.csv")
REPORTS_DIR = Path("reports")
SENTENCES_FILE = "sentences.csv"
ANNOTATIONS_FILE = "annotations.csv"
LEXICON_FILE = "lexicon.csv"

# Integer seed for reproducibility: same seed -> same reports.
SEED = 42

# Number of different reports written for each patient (row of the CSV).
# Raise it to produce more text for statistics.
VARIANTS_PER_PATIENT = 1

# How many absent symptoms / ruled-out diseases each report mentions (min, max).
N_ABSENT_SYMPTOMS = (1, 2)
N_RULED_OUT = (0, 1)

# Hard mode: also use templates where the negation comes AFTER the term,
# e.g. "Signs of paralysis were not observed."
POST_NEGATION = False

## Controlled vocabulary

Each canonical symptom / disease of the CSV has a small bank of synonyms (surface forms).
The banks were cleaned so that each surface form can be found as an **n-gram**:
no articles, no hyphens, and at most 5 tokens ("trembling on the little finger" is the longest).

The diagnosis `nothing` is **not** a disease: it is rendered by a sentence without any disease term.

In [2]:
SYMPTOM_PHRASES = {
    "paralysis": ["paralysis", "paralyzed limbs", "loss of motor control"],
    "yellow tongue": ["yellow tongue", "yellowish tongue", "yellow tinted tongue"],
    "trembling on the little finger": [
        "trembling on the little finger",
        "tremor in the little finger",
        "shaking of the pinky finger",
    ],
    "member loss": ["member loss", "limb loss", "missing limb"],
    "chest pain": ["chest pain", "pain in the chest", "thoracic discomfort"],
    "severe anger": ["severe anger", "intense anger episodes", "acute rage attacks"],
}

DISEASE_PHRASES = {
    "bacterial infection": [
        "bacterial infection",
        "infection of bacterial origin",
        "bacterial infectious process",
    ],
    "viral infection": [
        "viral infection",
        "infection of viral origin",
        "viral infectious process",
    ],
    "bite deficit": ["bite deficit", "bite related deficit", "deficit caused by bite"],
    "fights": ["fight injury", "fight related trauma", "altercation injury"],
}

PATHOGENS = ["Cyanobacterium zumbi", "Deltaretrovirus zumbi"]

# Negation cues and scope terminators (shared with Notebook 2 through the lexicon).
NEGATION_CUES = ["no", "not", "denies", "without", "absence", "negative", "free of", "ruled out", "absent"]
SCOPE_TERMINATORS = ["but", "however"]

NO_DISEASE = "nothing"

## Sentence templates

Each list holds templates of **one polarity**. `{t}` is the term; `{T}` is the term with a capital
initial letter. Templates never place an article or a verb that must agree in number with the term,
so any synonym fits any template.

| Section | Present | Absent |
|---|---|---|
| Presentation | present symptoms | — |
| Negative Findings | — | absent symptoms |
| Laboratory Analysis | pathogen found | no pathogen (term-free sentence) |
| Clinical History | previous diagnosis | no history |
| Assessment | diagnosis | ruled-out diseases |

In [3]:
SYMPTOM_PRESENT = [
    "The patient presents with {t}.",
    "Examination reveals {t}.",
    "The patient reports {t}.",
    "Findings include {t}.",
    "Signs of {t} are evident.",
]

SYMPTOM_ABSENT = [
    "No signs of {t}.",
    "The patient denies {t}.",
    "Examination reveals no {t}.",
    "The patient presents without {t}.",
    "Absence of {t}.",
    "The patient is free of {t}.",
]

# Hard mode only: the negation cue comes AFTER the term.
SYMPTOM_ABSENT_POST = [
    "Signs of {t} were not observed.",
    "{T}: absent.",
]

LAB_PRESENT = [
    "Blood analysis identified {t}.",
    "Laboratory tests isolated {t}.",
    "Blood culture is positive for {t}.",
]

LAB_ABSENT = [
    "No pathogen was isolated in the blood analysis.",
    "Blood analysis returned no relevant findings.",
    "Laboratory tests did not identify any pathogen.",
]

HISTORY_PRESENT = [
    "History of {t}.",
    "The patient has a history of {t}.",
    "Previous diagnosis of {t}.",
]

HISTORY_ABSENT = [
    "No history of {t}.",
    "The patient denies previous {t}.",
]

HISTORY_EMPTY = [
    "No previous diagnosis was recorded.",
    "Clinical history is unremarkable.",
]

DIAGNOSIS_PRESENT = [
    "Diagnosis: {t}.",
    "Clinical impression: {t}.",
    "The patient was diagnosed with {t}.",
    "Findings are consistent with {t}.",
]

DIAGNOSIS_NONE = [
    "No disease was identified.",
    "No diagnosis was established.",
]

DISEASE_RULED_OUT = [
    "No evidence of {t}.",
    "Negative for {t}.",
    "The team ruled out {t}.",
]

DISEASE_RULED_OUT_POST = [
    "{T} was ruled out.",
]

if POST_NEGATION:
    SYMPTOM_ABSENT = SYMPTOM_ABSENT + SYMPTOM_ABSENT_POST
    DISEASE_RULED_OUT = DISEASE_RULED_OUT + DISEASE_RULED_OUT_POST

## Read the source table

The CSV lists only the symptoms that are **present**. The **absent** symptoms and ruled-out diseases
are derived: absent symptoms are drawn from the symptoms *not* listed for that patient, and ruled-out
diseases favor diseases that share symptoms with the true diagnosis (a plausible differential).

In [4]:
patients = pd.read_csv(CSV_PATH).fillna("")

ALL_SYMPTOMS = list(SYMPTOM_PHRASES)
ALL_DISEASES = list(DISEASE_PHRASES)

def row_symptoms(row):
    return [row[f"Symptom {i}"] for i in (1, 2, 3) if row[f"Symptom {i}"]]

# Symptoms observed for each disease in the table (used to pick a plausible differential).
DISEASE_SYMPTOMS = {d: set() for d in ALL_DISEASES}
for _, row in patients.iterrows():
    if row["Diagnosis"] in DISEASE_SYMPTOMS:
        DISEASE_SYMPTOMS[row["Diagnosis"]].update(row_symptoms(row))

print(f"Read {len(patients)} patients.")
patients

Read 19 patients.


,Name,Diagnosis,Symptom 1,Symptom 2,Symptom 3,Blood Analysis,Previous Diagnosis
0,Rot Donnadd,bacterial infection,paralysis,yellow tongue,,Cyanobacterium zumbi,
1,Pid Mught,bacterial infection,yellow tongue,,,Cyanobacterium zumbi,
2,Thulk Lebbimp,bite deficit,yellow tongue,trembling on the little finger,,,bacterial infection
3,Bouvossam Damme,bite deficit,member loss,yellow tongue,trembling on the little finger,,bacterial infection
4,Pirg Zall,viral infection,chest pain,member loss,,Deltaretrovirus zumbi,
5,Nullon Rackindock,fights,member loss,severe anger,,,
6,Shor Splitturch,nothing,severe anger,,,,
7,Ger Ackeng,bite deficit,trembling on the little finger,,,,bacterial infection
8,Gleldo Shruck,bacterial infection,yellow tongue,chest pain,,Cyanobacterium zumbi,
9,Nadross Pilch,viral infection,chest pain,,,Deltaretrovirus zumbi,


## Generate the reports

Each report is a list of `(section, sentence, facts)` items, where `facts` records the term
mentioned in the sentence (surface form, canonical name, type, polarity). Randomness is seeded by
`(SEED, row, variant)`, so re-running reproduces exactly the same reports.

In [5]:
SECTIONS = ["Presentation", "Negative Findings", "Laboratory Analysis", "Clinical History", "Assessment"]

def slugify(name):
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")

def fill(template, term):
    return template.format(t=term, T=term[0].upper() + term[1:])

def mention(rng, templates, canonical, kind, polarity):
    """One sentence stating that a single term is present or absent."""
    if kind == "symptom":
        surface = rng.choice(SYMPTOM_PHRASES[canonical])
    elif kind == "disease":
        surface = rng.choice(DISEASE_PHRASES[canonical])
    else:
        surface = canonical
    sentence = fill(rng.choice(templates), surface)
    return sentence, [{"surface": surface, "canonical": canonical, "type": kind, "polarity": polarity}]

def sample_k(rng, population, k_range, weights=None):
    k = min(rng.randint(*k_range), len(population))
    chosen = []
    pool, w = list(population), list(weights) if weights else [1] * len(population)
    for _ in range(k):
        pick = rng.choices(range(len(pool)), weights=w, k=1)[0]
        chosen.append(pool.pop(pick))
        w.pop(pick)
    return chosen

def generate_report(row, index, variant):
    rng = random.Random(f"{SEED}-{index}-{variant}")
    items = []

    present = row_symptoms(row)
    for s in present:
        items.append(("Presentation", *mention(rng, SYMPTOM_PRESENT, s, "symptom", "present")))

    candidates = [s for s in ALL_SYMPTOMS if s not in present]
    for s in sample_k(rng, candidates, N_ABSENT_SYMPTOMS):
        items.append(("Negative Findings", *mention(rng, SYMPTOM_ABSENT, s, "symptom", "absent")))

    if row["Blood Analysis"]:
        items.append(("Laboratory Analysis", *mention(rng, LAB_PRESENT, row["Blood Analysis"], "pathogen", "present")))
    else:
        items.append(("Laboratory Analysis", rng.choice(LAB_ABSENT), []))

    diagnosis = row["Diagnosis"]
    if row["Previous Diagnosis"]:
        items.append(("Clinical History", *mention(rng, HISTORY_PRESENT, row["Previous Diagnosis"], "disease", "present")))
    elif rng.random() < 0.5:
        items.append(("Clinical History", rng.choice(HISTORY_EMPTY), []))
    else:
        denied = rng.choice([d for d in ALL_DISEASES if d != diagnosis])
        items.append(("Clinical History", *mention(rng, HISTORY_ABSENT, denied, "disease", "absent")))

    if diagnosis == NO_DISEASE:
        items.append(("Assessment", rng.choice(DIAGNOSIS_NONE), []))
    else:
        items.append(("Assessment", *mention(rng, DIAGNOSIS_PRESENT, diagnosis, "disease", "present")))

    others = [d for d in ALL_DISEASES if d != diagnosis]
    weights = [1 + len(DISEASE_SYMPTOMS[d] & set(present)) for d in others]
    for d in sample_k(rng, others, N_RULED_OUT, weights):
        items.append(("Assessment", *mention(rng, DISEASE_RULED_OUT, d, "disease", "absent")))

    return items

def render_markdown(report_id, name, items):
    lines = ["# Zombie Clinical Report", "", f"**Case ID:** {report_id}", f"**Patient:** {name}"]
    for section in SECTIONS:
        sentences = [sentence for sec, sentence, _ in items if sec == section]
        if sentences:
            lines += ["", f"## {section}", " ".join(sentences)]
    return "\n".join(lines) + "\n"

In [6]:
REPORTS_DIR.mkdir(exist_ok=True)
for old_file in REPORTS_DIR.glob("*.md"):
    old_file.unlink()

sentence_rows, annotation_rows = [], []

for index, row in patients.iterrows():
    for variant in range(1, VARIANTS_PER_PATIENT + 1):
        report_id = f"ZH-{index + 1:03d}"
        file_name = slugify(row["Name"])
        if VARIANTS_PER_PATIENT > 1:
            report_id += f"-{variant}"
            file_name += f"-{variant}"

        items = generate_report(row, index + 1, variant)

        for n, (section, sentence, facts) in enumerate(items, start=1):
            sentence_id = f"{report_id}-S{n:02d}"
            sentence_rows.append({"report_id": report_id, "section": section,
                                  "sentence_id": sentence_id, "sentence": sentence})
            for fact in facts:
                annotation_rows.append({"report_id": report_id, "sentence_id": sentence_id, **fact})

        (REPORTS_DIR / f"{file_name}.md").write_text(render_markdown(report_id, row["Name"], items))

sentences = pd.DataFrame(sentence_rows)
annotations = pd.DataFrame(annotation_rows)

sentences.to_csv(SENTENCES_FILE, index=False)
annotations.to_csv(ANNOTATIONS_FILE, index=False)

print(f"Generated {sentences['report_id'].nunique()} reports in {REPORTS_DIR}/")
print(f"Saved {len(sentences)} sentences to {SENTENCES_FILE}")
print(f"Saved {len(annotations)} annotated facts to {ANNOTATIONS_FILE}")

Generated 19 reports in reports/
Saved 122 sentences to sentences.csv
Saved 105 annotated facts to annotations.csv


## Save the lexicon (the "field guide")

The lexicon lists every expression a simple algorithm is allowed to know about, with its category.
Notebook 2 builds its dictionaries from this file.

In [7]:
lexicon_rows = (
    [{"surface": s, "canonical": c, "type": "symptom"} for c, ss in SYMPTOM_PHRASES.items() for s in ss]
    + [{"surface": s, "canonical": c, "type": "disease"} for c, ss in DISEASE_PHRASES.items() for s in ss]
    + [{"surface": p, "canonical": p, "type": "pathogen"} for p in PATHOGENS]
    + [{"surface": w, "canonical": w, "type": "negation"} for w in NEGATION_CUES]
    + [{"surface": w, "canonical": w, "type": "terminator"} for w in SCOPE_TERMINATORS]
)
lexicon = pd.DataFrame(lexicon_rows)
lexicon["surface"] = lexicon["surface"].str.lower()
lexicon.to_csv(LEXICON_FILE, index=False)

lexicon.groupby("type").size()

type
disease       12
negation       9
pathogen       2
symptom       18
terminator     2
dtype: int64

## Sanity checks

The generated facts must agree with the source table:

* every symptom listed in the CSV appears as **present** in that patient's report;
* no **absent** symptom belongs to the patient's row;
* the diagnosis appears as **present** (except for `nothing`) and is never ruled out.

In [8]:
for index, row in patients.iterrows():
    for report_id, facts in annotations[annotations["report_id"].str.startswith(f"ZH-{index + 1:03d}")].groupby("report_id"):
        symptoms = facts[facts["type"] == "symptom"]
        present = set(symptoms.loc[symptoms["polarity"] == "present", "canonical"])
        absent = set(symptoms.loc[symptoms["polarity"] == "absent", "canonical"])
        assert present == set(row_symptoms(row)), report_id
        assert not absent & set(row_symptoms(row)), report_id

        assessment = facts.merge(sentences, on=["report_id", "sentence_id"])
        assessment = assessment[assessment["section"] == "Assessment"]
        diagnosed = set(assessment.loc[assessment["polarity"] == "present", "canonical"])
        ruled_out = set(assessment.loc[assessment["polarity"] == "absent", "canonical"])
        expected = set() if row["Diagnosis"] == NO_DISEASE else {row["Diagnosis"]}
        assert diagnosed == expected, report_id
        assert row["Diagnosis"] not in ruled_out, report_id

print("All checks passed.")

All checks passed.


In [9]:
# Balance of the facts: how many present / absent mentions of each type.
pd.crosstab(annotations["type"], annotations["polarity"], margins=True)

polarity,absent,present,All
type,,,
disease,15,25,40
pathogen,0,8,8
symptom,26,31,57
All,41,64,105


In [10]:
# Two example reports.
for path in sorted(REPORTS_DIR.glob("*.md"))[:2]:
    print(path.read_text())

# Zombie Clinical Report

**Case ID:** ZH-015
**Patient:** Blottork Patter

## Presentation
Findings include yellow tongue. The patient presents with trembling on the little finger.

## Negative Findings
Absence of severe anger.

## Laboratory Analysis
Blood analysis returned no relevant findings.

## Clinical History
Previous diagnosis of infection of bacterial origin.

## Assessment
Diagnosis: bite related deficit. The team ruled out bacterial infectious process.

# Zombie Clinical Report

**Case ID:** ZH-004
**Patient:** Bouvossam Damme

## Presentation
Findings include member loss. Examination reveals yellowish tongue. Findings include shaking of the pinky finger.

## Negative Findings
The patient denies severe anger. No signs of thoracic discomfort.

## Laboratory Analysis
Blood analysis returned no relevant findings.

## Clinical History
History of bacterial infectious process.

## Assessment
Diagnosis: deficit caused by bite.



In [11]:
# The sentences of one report, with the facts they carry.
first = sentences["report_id"].iloc[0]
sentences[sentences["report_id"] == first].merge(
    annotations, on=["report_id", "sentence_id"], how="left"
)[["section", "sentence", "surface", "canonical", "type", "polarity"]]

,section,sentence,surface,canonical,type,polarity
0,Presentation,The patient presents with paralysis.,paralysis,paralysis,symptom,present
1,Presentation,Findings include yellow tinted tongue.,yellow tinted tongue,yellow tongue,symptom,present
2,Negative Findings,The patient denies missing limb.,missing limb,member loss,symptom,absent
3,Laboratory Analysis,Blood analysis identified Cyanobacterium zumbi.,Cyanobacterium zumbi,Cyanobacterium zumbi,pathogen,present
4,Clinical History,No previous diagnosis was recorded.,NaN,NaN,NaN,NaN
5,Assessment,Diagnosis: bacterial infection.,bacterial infection,bacterial infection,disease,present
